In [1]:
from pathlib import Path
import sys

# --- Paths / imports -------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
PREPROCESSING_DIR = PROJECT_ROOT / "functions" / "preprocessing"
for p in (PROJECT_ROOT, PREPROCESSING_DIR):
    if str(p) not in sys.path:
        sys.path.append(str(p))

import pickle
from server_config import redcap_path, preprocessed_path

import pandas as pd
import numpy as np
import datetime as dt

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel
import plotly.graph_objects as go


In [2]:
# Import preprocessed data

with open(preprocessed_path + '/ema_content.pkl', 'rb') as file:
    df_ema_content = pickle.load(file)  
    
with open(preprocessed_path + '/ema_meta.pkl', 'rb') as file:
    df_ema_meta = pickle.load(file) 
    
with open(preprocessed_path + '/monitoring_data.pkl', 'rb') as file:
    df_monitoring = pickle.load(file)
    
df_sp6_meta = pd.read_csv(preprocessed_path + '/meta/SP6_meta_data.csv')
    
df_redcap_final = pd.read_spss(redcap_path + '/sp1_cleaned' + '/baseline_raw_20251012.sav')

df_steps = pd.read_parquet(preprocessed_path + "/passive/daily/daily_agg_steps.parquet")


In [3]:
# Filter variables
# Minimum beeps per burst see: https://pmc.ncbi.nlm.nih.gov/articles/PMC12683972/pdf/nihms-2115745.pdf 

measurement_bursts = [0]
min_beeps = 10 

## 0. Prepare Sample 

In [4]:
df_ema_short = df_ema_content[["id", "for_id","measurement_burst", "timestamp_beep_completion", "item", "quest_nr_str","response",
                                "date","beep_per_person_id", "weekday","n_beeps_completed_per_burst"]].copy()

In [5]:
df_ema_short = df_ema_short[df_ema_short["measurement_burst"].isin(measurement_bursts)].copy()
df_ema_short_filtered = df_ema_short[df_ema_short["n_beeps_completed_per_burst"] >= min_beeps].copy()

In [6]:
df_ema_content = df_ema_short_filtered.copy()

In [7]:
df_ema_info = df_ema_content[["id", "for_id", "n_beeps_completed_per_burst"]].drop_duplicates()

## 1. Affect 

In [8]:
df_ema_panas = df_ema_content[df_ema_content["item"].str.startswith("panas_", na=False)]

extra_cols = ["measurement_burst", "timestamp_beep_completion", "quest_nr_str", "n_beeps_completed_per_burst"]

# Pivot the table as specified
df_panas_piv = df_ema_panas.pivot_table(
    index=["id", "beep_per_person_id", "measurement_burst"],
    columns="item",
    values="response",
    aggfunc="first"  # Using 'first' since each entry should theoretically be unique per group
)

# The columns are now a single level Index with just the quest_title values since 'values' is not a list anymore
df_panas_piv.columns = [col for col in df_panas_piv.columns.values]

# Reset the index to turn the MultiIndex into columns
df_panas_piv = df_panas_piv.reset_index()
df_panas_piv = df_panas_piv.drop_duplicates()

In [9]:
na_cols = ['panas_fear1', 'panas_fear2', 'panas_guilt1','panas_guilt2', 'panas_hostility1', 'panas_hostility2','panas_sadness1', 'panas_sadness2','panas_loneliness']
pa_cols = ['panas_attentiveness', 'panas_joviality1', 'panas_joviality2','panas_selfassurance','panas_serenity1', 'panas_serenity2']

sadness_cols = ['panas_sadness1', 'panas_sadness2']
fear_cols = ['panas_fear1', 'panas_fear2']
hostility_cols = ['panas_hostility1', 'panas_hostility2']
guilt_cols = ['panas_guilt1','panas_guilt2']

joviality_cols = ['panas_joviality1', 'panas_joviality2']
serenity_cols = ['panas_serenity1', 'panas_serenity2']

pa_high_cols = ["panas_attentiveness","panas_joviality1", "panas_joviality2"]
pa_low_cols = ["panas_serenity1", "panas_serenity2"]

na_high_cols = ["panas_fear1", "panas_fear2","panas_hostility1", "panas_hostility2"]
na_low_cols = ["panas_sadness1", "panas_sadness2", "panas_fatigue"]


In [10]:
# all item columns used anywhere
panas_all_cols = sorted(set(
    na_cols
    + pa_cols
    + sadness_cols
    + fear_cols
    + hostility_cols
    + guilt_cols
    + joviality_cols
    + serenity_cols
    + pa_high_cols
    + pa_low_cols
    + na_high_cols
    + na_low_cols
))

# make sure they are numeric
df_panas_piv[panas_all_cols] = df_panas_piv[panas_all_cols].apply(
    pd.to_numeric, errors="coerce"
)


In [11]:
group_cols = ['id', 'beep_per_person_id', 'measurement_burst']

scale_defs = {
    'panas_na':        na_cols,
    'panas_pa':        pa_cols,
    'panas_pa_high_ar':pa_high_cols,
    'panas_pa_low_ar': pa_low_cols,
    'panas_na_high_ar':na_high_cols,
    'panas_na_low_ar': na_low_cols,
    'panas_sadness':   sadness_cols,
    'panas_fear':      fear_cols,
    'panas_hostility': hostility_cols,
    'panas_guilt':     guilt_cols,
    'panas_joviality': joviality_cols,
    'panas_serenity':  serenity_cols,
}

for new_col, cols in scale_defs.items():
    df_panas_piv[new_col] = (
        df_panas_piv
        .groupby(group_cols)[cols]
        .transform('mean')
        .mean(axis=1)
    )


### 1.1 Mean and standard deviation

In [12]:
col_list = ['panas_na','panas_pa','panas_pa_high_ar','panas_pa_low_ar','panas_na_high_ar','panas_na_low_ar','panas_sadness','panas_fear','panas_hostility','panas_guilt','panas_joviality',
            'panas_serenity', 'panas_loneliness', 'panas_fatigue','panas_shyness', 'panas_attentiveness', 'panas_selfassurance']

In [13]:

df_panas_piv[col_list] = df_panas_piv[col_list].apply(pd.to_numeric, errors='coerce')

agg_affect = (
    df_panas_piv
    .groupby(['id', 'measurement_burst'])[col_list]
    .agg(['mean', 'std'])
)

agg_affect.columns = [
    f"{col}_{stat if stat != 'std' else 'sd'}"
    for col, stat in agg_affect.columns
]

agg_affect = agg_affect.reset_index()

In [14]:
df_affect_sam = agg_affect[['id', 'panas_na_mean', 'panas_na_sd', 'panas_pa_mean', 'panas_pa_sd', 'panas_sadness_mean', 'panas_sadness_sd',
       'panas_fear_mean', 'panas_fear_sd', 'panas_hostility_mean','panas_hostility_sd', 'panas_guilt_mean', 'panas_guilt_sd',
       'panas_joviality_mean', 'panas_joviality_sd', 'panas_serenity_mean','panas_serenity_sd', 'panas_loneliness_mean', 'panas_loneliness_sd',
       'panas_fatigue_mean', 'panas_fatigue_sd', 'panas_shyness_mean','panas_shyness_sd', 'panas_attentiveness_mean',
       'panas_attentiveness_sd', 'panas_selfassurance_mean','panas_selfassurance_sd']]

### 1.2 Emotion differentiation

In [15]:
na_items = ['panas_fear', 'panas_guilt', 'panas_hostility','panas_sadness','panas_loneliness', 'panas_fatigue']
pa_items = ['panas_attentiveness', 'panas_joviality','panas_selfassurance','panas_serenity1']
all_items = na_items + pa_items
df_panas_piv[all_items] = df_panas_piv[all_items].apply(pd.to_numeric, errors='coerce')


In [16]:

def compute_momentary_ed(df, emotion_cols, id_col="customer", phase_col="assess", prefix="na"):
    """
    Momentary Emotion Differentiation (EDi) nach Erbas et al. (2021),
    getrennt pro Person × Phase und getrennt für ein Set von Emotionen (NA oder PA).

    Neue Spalten im zurückgegebenen df:
      {prefix}_EDi        : momentary ED pro Beep (<= 0; näher 0 = höhere Differentiation)
      {prefix}_ED_overall : Aggregat pro Person × Phase (aus nonED_i, analog zum overall Index)
      {prefix}_n_beeps    : Anzahl gültiger Beeps in dieser Person × Phase
    """
    df = df.copy()

    # Nur die Emotionsspalten verwenden → Gruppierungsspalten sind keine DataFrame-Spalten mehr
    emo = df[emotion_cols].astype(float)

    J = len(emotion_cols)

    def _per_group(g):
        # g: DataFrame mit nur den Emotionsspalten, Index = ursprünglicher df-Index
        valid_rows = ~g.isna().all(axis=1)
        emo_valid = g.loc[valid_rows]
        idx_valid = emo_valid.index
        n = emo_valid.shape[0]

        # Default: alles NaN
        ed = pd.Series(np.nan, index=g.index)
        overall = pd.Series(np.nan, index=g.index)
        n_beeps = pd.Series(n, index=g.index)  # n pro Gruppe, in jede Zeile geschrieben

        if n <= 1:
            return pd.DataFrame({
                f"{prefix}_EDi": ed,
                f"{prefix}_ED_overall": overall,
                f"{prefix}_n_beeps": n_beeps
            }, index=g.index)

        # (1) person-mean zentrieren (innerhalb der Person×Phase-Gruppe)
        centered = emo_valid - emo_valid.mean(axis=0)

        # (4)+(5) Varianzen pro Item und Summe über Items
        denom = centered.var(ddof=1, axis=0).sum()

        if denom <= 0 or np.isnan(denom):
            return pd.DataFrame({
                f"{prefix}_EDi": ed,
                f"{prefix}_ED_overall": overall,
                f"{prefix}_n_beeps": n_beeps
            }, index=g.index)

        # (2) Mittelwert der zentrierten Items pro Beep
        row_mean = centered.mean(axis=1)

        # (3) Zähler: (J * mean)^2
        num = (row_mean * J) ** 2

        # nonED_i & EDi
        noned = num / denom
        ed.loc[idx_valid] = -noned  # näher 0 = höhere Differentiation

        # Aggregierter ED pro Person×Phase
        overall_val = -noned.sum() / (n - 1)
        overall[:] = overall_val

        return pd.DataFrame({
            f"{prefix}_EDi": ed,
            f"{prefix}_ED_overall": overall,
            f"{prefix}_n_beeps": n_beeps
        }, index=g.index)

    # Gruppierung über externe Keys (df[id_col], df[phase_col]) → keine Gruppierungsspalten in g
    res = emo.groupby([df[id_col], df[phase_col]], group_keys=False).apply(_per_group)

    # Ergebnisse über den Index mit df alignen
    df[[f"{prefix}_EDi", f"{prefix}_ED_overall", f"{prefix}_n_beeps"]] = res[
        [f"{prefix}_EDi", f"{prefix}_ED_overall", f"{prefix}_n_beeps"]
    ]

    return df


In [17]:
# Zuerst NA-ED
df_ed = compute_momentary_ed(
    df_panas_piv,
    emotion_cols=na_items,
    id_col="id",
    phase_col="measurement_burst",
    prefix="na"
)

# Danach PA-ED (baut auf df_ed auf, fügt nur neue Spalten hinzu)
df_ed = compute_momentary_ed(
    df_ed,
    emotion_cols=pa_items,
    id_col="id",
    phase_col="measurement_burst",
    prefix="pa"
)


In [18]:
def compute_icc_emodiff(df, emotion_cols, id_col="customer", phase_col="assess", prefix="na"):
    """
    Klassischer ICC/Cronbach's-Alpha-basierter Differentiationsindex pro Person × Phase.

    Neue Spalten im zurückgegebenen df:
      {prefix}_ICC_raw      : ICC/Cronbach's alpha (0–1, höher = mehr Kovariation = weniger Differentiation)
      {prefix}_ICC_diff     : 1 - ICC_raw (0–1, höher = mehr Differentiation)
      {prefix}_n_beeps_icc  : Anzahl gültiger Beeps in dieser Person × Phase
    """
    df = df.copy()

    emo = df[emotion_cols].astype(float)
    J = len(emotion_cols)

    def _icc_group(g):
        # g: nur Emotionsspalten, Index = ursprünglicher df-Index
        valid = ~g.isna().all(axis=1)
        emo_valid = g.loc[valid]
        n = emo_valid.shape[0]

        icc_raw = np.nan

        if n > 1 and J > 1:
            item_vars = emo_valid.var(ddof=1)          # Varianz pro Item
            sum_item_vars = item_vars.sum()

            sum_scores = emo_valid.sum(axis=1)         # Sumscore pro Beep
            total_var = sum_scores.var(ddof=1)

            if total_var > 0 and sum_item_vars >= 0:
                icc_raw = (J / (J - 1.0)) * (1.0 - (sum_item_vars / total_var))
                if icc_raw < 0:
                    icc_raw = np.nan

        icc_raw_series = pd.Series(icc_raw, index=g.index)
        icc_diff_series = pd.Series(
            (1.0 - icc_raw) if pd.notna(icc_raw) else np.nan,
            index=g.index
        )
        n_series = pd.Series(n, index=g.index)

        return pd.DataFrame({
            f"{prefix}_ICC_raw": icc_raw_series,
            f"{prefix}_ICC_diff": icc_diff_series,
            f"{prefix}_n_beeps_icc": n_series
        }, index=g.index)

    res = emo.groupby([df[id_col], df[phase_col]], group_keys=False).apply(_icc_group)

    df[[f"{prefix}_ICC_raw", f"{prefix}_ICC_diff", f"{prefix}_n_beeps_icc"]] = res[
        [f"{prefix}_ICC_raw", f"{prefix}_ICC_diff", f"{prefix}_n_beeps_icc"]
    ]

    return df


In [19]:
# 1) ICC für negative Emotionen pro Person × Phase
df_icc = compute_icc_emodiff(
    df=df_panas_piv,
    emotion_cols=na_items,
    id_col="id",
    phase_col="measurement_burst",
    prefix="na"
)

# 2) ICC für positive Emotionen pro Person × Phase
df_icc = compute_icc_emodiff(
    df=df_icc,
    emotion_cols=pa_items,
    id_col="id",
    phase_col="measurement_burst",
    prefix="pa"
)

In [20]:
# Phase-Level-ICC (eine Zeile pro id × measurement_burst)
icc_phase = (
    df_icc
    .drop_duplicates(subset=["id", "measurement_burst"])
    [["id", "measurement_burst","beep_per_person_id",
      "na_ICC_raw", "na_ICC_diff", "na_n_beeps_icc",
      "pa_ICC_raw", "pa_ICC_diff"]]
    .copy()
)


In [21]:
# Merge PANAS-EMA mit momentary und aggregierten ED-Maßen
df_merged = agg_affect.merge(
    df_ed[
        ["id", "measurement_burst","beep_per_person_id",
         "na_EDi", "na_ED_overall",
         "pa_EDi", "pa_ED_overall"]
    ],
    on=["id", "measurement_burst"],
    how="left"
)

df_affect_final = df_merged.merge(
    icc_phase,
    on=["id", "measurement_burst", "beep_per_person_id"],
    how="left"
)


In [22]:
df_monitoring_final = df_affect_final.merge(df_monitoring, on = "id", how= "left")

### 2. Emotion Regulation

In [23]:
df_ema_er = df_ema_content[df_ema_content["item"].str.startswith("er_", na=False)]

extra_cols = ["measurement_burst", "timestamp_beep_completion", "quest_nr_str"]

# Pivot the table as specified
df_er_piv = df_ema_er.pivot_table(
    index=["id", "beep_per_person_id", "measurement_burst"],
    columns="item",
    values="response",
    aggfunc='first'  # Using 'first' since each entry should theoretically be unique per group
)

# The columns are now a single level Index with just the quest_title values since 'values' is not a list anymore
df_er_piv.columns = [col for col in df_er_piv.columns.values]

# Reset the index to turn the MultiIndex into columns
df_er_piv = df_er_piv.reset_index()
df_er_piv = df_er_piv.drop_duplicates()

In [24]:
col_list_er = ['er_acceptance','er_control', 'er_distraction','er_intensity','er_reappraisal','er_relaxation','er_rumination','er_suppression']

In [25]:

df_er_piv[col_list_er] = df_er_piv[col_list_er].apply(pd.to_numeric, errors='coerce')

agg_er = (
    df_er_piv
    .groupby(['id', 'measurement_burst'])[col_list_er]
    .agg(['mean', 'std'])
)

agg_er.columns = [
    f"{col}_{stat if stat != 'std' else 'sd'}"
    for col, stat in agg_er.columns
]

agg_er = agg_er.reset_index()

#### Export features for Sam

In [26]:
df_er_sam = agg_er[['id','er_acceptance_mean', 'er_acceptance_sd',
       'er_control_mean', 'er_control_sd', 'er_distraction_mean',
       'er_distraction_sd', 'er_intensity_mean', 'er_intensity_sd',
       'er_reappraisal_mean', 'er_reappraisal_sd', 'er_relaxation_mean',
       'er_relaxation_sd', 'er_rumination_mean', 'er_rumination_sd',
       'er_suppression_mean', 'er_suppression_sd']]

In [27]:

# ------------------------------------------------------------------
# 2) Moment-level indices (per beep)
#    - between-strategy SD at each beep
#    - mean ER-endorsement
#    - repertoire size (optional)
# ------------------------------------------------------------------

# SD across strategies within a beep = between-strategy variability (momentary)
df_er_piv["er_between_sd_beep"] = df_er_piv[col_list_er].std(axis=1, ddof=1)

# mean endorsement within a beep
df_er_piv["er_mean_endorse_beep"] = df_er_piv[col_list_er].mean(axis=1)

# number of strategies "used" at this beep (e.g. > 0)
df_er_piv["er_n_strategies_beep"] = (df_er_piv[col_list_er] > 1).sum(axis=1)

# ------------------------------------------------------------------
# 3) Person × phase indices
#    - mean between-strategy SD over beeps
#    - within-strategy SD over time, averaged across strategies
#    - mean overall ER endorsement
#    - number of beeps
# ------------------------------------------------------------------

# 3a) Between-strategy variability: mean SD_across_strategies over beeps
er_between_phase = (
    df_er_piv
    .groupby(["id", "measurement_burst"], as_index=False)["er_between_sd_beep"]
    .mean()
    .rename(columns={"er_between_sd_beep": "er_between_sd_mean"})
)

# 3b) Within-strategy variability:
#     SD over beeps for each strategy, then mean across strategies
er_within_phase = (
    df_er_piv
    .groupby(["id", "measurement_burst"])[col_list_er]
    .std(ddof=1)           # SD over time per strategy
    .reset_index()
)

er_within_phase["er_within_sd_mean"] = er_within_phase[col_list_er].mean(axis=1)

# 3c) Mean overall ER endorsement per person×phase
er_mean_phase = (
    df_er_piv
    .groupby(["id", "measurement_burst"])[col_list_er]
    .mean()
    .reset_index()
)

er_mean_phase["er_mean_overall"] = er_mean_phase[col_list_er].mean(axis=1)

# 3d) Number of beeps per person×phase
n_beeps_phase = (
    df_er_piv
    .groupby(["id", "measurement_burst"], as_index=False)
    .size()
    .rename(columns={"size": "er_n_beeps"})
)

er_rep_phase = (
    df_er_piv
    .groupby(["id", "measurement_burst"], as_index=False)["er_n_strategies_beep"]
    .mean()
    .rename(columns={"er_n_strategies_beep": "er_n_strategies_mean"})
)

er_flex_phase = (
    er_between_phase
    .merge(er_within_phase[["id", "measurement_burst", "er_within_sd_mean"]],
           on=["id", "measurement_burst"], how="left")
    .merge(er_mean_phase[["id", "measurement_burst", "er_mean_overall"]],
           on=["id", "measurement_burst"], how="left")
    .merge(n_beeps_phase, on=["id", "measurement_burst"], how="left")
    .merge(er_rep_phase, on=["id", "measurement_burst"], how="left")
)


In [28]:
df_er_final = er_flex_phase.merge(agg_er, on= ["id", "measurement_burst"])

### 3. Physical Health

In [29]:
df_ema_ph = df_ema_content[df_ema_content["item"] == "physical_health"]

extra_cols = ["measurement_burst", "timestamp_beep_completion", "quest_nr_str", "n_beeps_completed_per_burst"]

# Pivot the table as specified
df_ph_piv = df_ema_ph.pivot_table(
    index=["id", "beep_per_person_id", "measurement_burst"],
    columns="item",
    values="response",
    aggfunc='first'  # Using 'first' since each entry should theoretically be unique per group
)

# The columns are now a single level Index with just the quest_title values since 'values' is not a list anymore
df_ph_piv.columns = [col for col in df_ph_piv.columns.values]

# Reset the index to turn the MultiIndex into columns
df_ph_piv = df_ph_piv.reset_index()
df_ph_piv = df_ph_piv.drop_duplicates()

In [30]:
# make sure they are numeric
df_ph_piv["physical_health"] = df_ph_piv["physical_health"].apply(
    pd.to_numeric, errors="coerce"
)

In [31]:
agg_ph = (
    df_ph_piv
    .groupby(['id', 'measurement_burst'])["physical_health"]
    .agg(['mean', 'std'])
)

agg_ph = agg_ph.reset_index()
agg_ph.rename({"mean":"ph_mean","std":"ph_std"}, axis=1, inplace=True)

### 4. Social Contact Valence

In [32]:
df_ema_social = df_ema_content[df_ema_content["item"] == "event_social3"]

extra_cols = ["measurement_burst", "timestamp_beep_completion", "quest_nr_str", "n_beeps_completed_per_burst"]

# Pivot the table as specified
df_social_piv = df_ema_social.pivot_table(
    index=["id", "beep_per_person_id", "measurement_burst"],
    columns="item",
    values="response",
    aggfunc='first'  # Using 'first' since each entry should theoretically be unique per group
)

# The columns are now a single level Index with just the quest_title values since 'values' is not a list anymore
df_social_piv.columns = [col for col in df_social_piv.columns.values]

# Reset the index to turn the MultiIndex into columns
df_social_piv = df_social_piv.reset_index()
df_social_piv = df_social_piv.drop_duplicates()

In [33]:
df_ema_social = df_ema_content[df_ema_content["item"] == "event_social3"]

extra_cols = ["measurement_burst", "timestamp_beep_completion", "quest_nr_str", "n_beeps_completed_per_burst"]

# Pivot the table as specified
df_social_piv = df_ema_social.pivot_table(
    index=["id", "beep_per_person_id", "measurement_burst"],
    columns="item",
    values="response",
    aggfunc='first'  # Using 'first' since each entry should theoretically be unique per group
)

# The columns are now a single level Index with just the quest_title values since 'values' is not a list anymore
df_social_piv.columns = [col for col in df_social_piv.columns.values]

# Reset the index to turn the MultiIndex into columns
df_social_piv = df_social_piv.reset_index()
df_social_piv = df_social_piv.drop_duplicates()

In [34]:
# make sure they are numeric
df_social_piv["event_social3"] = df_social_piv["event_social3"].apply(
    pd.to_numeric, errors="coerce"
)

In [35]:
agg_social = (
    df_social_piv
    .groupby(['id', 'measurement_burst'])["event_social3"]
    .agg(['mean', 'std'])
)

agg_social = agg_social.reset_index()
agg_social.rename({"mean":"social_valence_mean","std":"social_valence_std"}, axis=1, inplace=True)

### 5. Meta Variables 

In [36]:
rename_columns = {'ema_smartphone': 'smartphone_version',
                  'ema_wear_exp': 'wearable_experience',
                  'ema_sleep': 'chronotype',
                  'ema_special_event': 'special_event_during_ema',
                  'ema_phonecall': 'onboarding_call_happened'
                 }

# apply renaming
df_sp6_meta = df_sp6_meta.rename(columns=rename_columns)

In [37]:
df_sp6_meta_sam = df_sp6_meta[['for_id','chronotype', 'special_event_during_ema', 'onboarding_call_happened',
                               'ema_base_start', 'ema_base_end']]

In [38]:
df_sam = df_sp6_meta_sam.merge(df_ema_info, on="for_id", how="right")

In [39]:
df_sam = df_sam.merge(df_er_sam, on= "id")

In [40]:
df_sam = df_sam.merge(df_affect_sam, on= "id")

In [41]:
df_sam.to_csv(preprocessed_path + "/crosssectional/" + "features_ema_sam.csv")

### 6. Passive Data

In [42]:
df_steps_short = df_steps[["for_id","local_day", "StepsInDay"]]

In [43]:
df_steps_short = df_steps_short.merge(df_sp6_meta, on= "for_id")

In [44]:
df_steps_short['ema_base_end'] = pd.to_datetime(df_steps_short['ema_base_end'])
df_steps_short['ema_base_end'] = df_steps_short['ema_base_end'].dt.tz_localize(None)

# Fix ema_base_start timezone
df_steps_short['ema_base_start'] = pd.to_datetime(df_steps_short['ema_base_start'])
df_steps_short['ema_base_start'] = df_steps_short['ema_base_start'].dt.tz_localize(None)

# Filter to only days within the EMA base window
df_steps_short = df_steps_short[
    (df_steps_short.local_day >= df_steps_short.ema_base_start) &
    (df_steps_short.local_day <= df_steps_short.ema_base_end)
]

df_person_summary = df_steps_short.groupby('for_id').agg(
    day_count=('local_day', 'count'),
    steps_mean=('StepsInDay', 'mean'),
    steps_sd=('StepsInDay', 'std')
).reset_index()

# Binary variable for >= 7 days
df_person_summary['sufficient_days_steps'] = (df_person_summary['day_count_steps'] > 6).astype(int)

In [46]:
df_sam = df_sam.merge(df_summary_steps, on="for_id", how="left")

In [50]:
df_sam.to_csv(preprocessed_path + "/crosssectional/" + "features_ema_sam.csv")